# 02 · Train the Random Forest (v1, per-packet features)

Trains the `rf` model the backend loads from `models/rf/`, on the **train** files of the prepared CIC-IDS2017 dataset (`01_prepare`). It picks a threshold on the **val** files and exports with `nla_ml.inference.export_model`.

**It never reads the test files.** Test results belong to `04_compare`, so nothing here is tuned on them.

| Step | What | Why |
|---|---|---|
| Load | Featurize each train file, collapse identical (feature row, attack) pairs into one row with a `count` | 24 M packets become a few million distinct rows. `count` becomes the sample weight, so the model still sees the real packet mix |
| Weights | `count × class balance`, computed from **packet** counts | `class_weight="balanced"` would balance distinct rows, not packets |
| Tune | Small grid, leave-one-**day**-out CV over the train files | Each fold scores a model on a capture day it never saw. Criterion: **macro per-family recall** at `TARGET_FPR`, so Hulk/DDoS volume doesn't decide everything |
| Fit | Best config, more trees, on all train data | |
| Threshold | Score at which `TARGET_FPR` of val's ~7 M benign packets are flagged | Stored in the metadata. The backend itself uses the score through the noisy-OR weight |
| Export | `export_model` → reload with `load_model` → latency per 500-row batch | Proves the artifact passes every backend check before you download it |

**Out** (`/kaggle/working/`): `models/rf/{model.joblib, metadata.json}`, `cv_results.csv`, `report.json`.

**Inputs:** add the prepared dataset (e.g. `hannana06/nla-cicids2017-prepared`) to this notebook. Internet must be on for the pip install. Remember what to expect: the separability ceilings say per-packet features can't catch most DDoS, Slowhttptest or XSS packets.

In [ ]:
!pip install -q "nla-ml[sklearn] @ git+https://github.com/HannanAFC/NetLogAnalyser.git@2aa7e428e66d890bf325a23fa33fce0a00a87bac#subdirectory=backend/ml"
# Restart the kernel after the first install (it replaces Kaggle's scikit-learn), then run from the next cell.

In [ ]:
import gc
import json
import time
from datetime import UTC, datetime
from pathlib import Path

import nla_ml
import numpy as np
import pandas as pd
from nla_ml.adapters.cicids2017_packets import BENIGN_LABEL, ENTRY_COLUMNS, read_prepared
from nla_ml.compat import SKLEARN_VERSION, assert_sklearn_version
from nla_ml.features import FEATURE_COLUMNS, FEATURE_FINGERPRINT, FEATURE_VERSION, featurize, featurize_frame
from nla_ml.inference import export_model, load_model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

assert_sklearn_version( )   # fails if the kernel wasn't restarted after the install

NLA_ML_COMMIT   = "2aa7e428e66d890bf325a23fa33fce0a00a87bac"   # keep equal to the pip install pin
DATASET_VERSION = ""        # the version of the prepared Kaggle dataset you attached, e.g. "hannana06/nla-cicids2017-prepared:1"
INPUT           = next( Path( "/kaggle/input" ).rglob( "manifest.json" ) ).parent
OUT             = Path( "/kaggle/working" )
TARGET_FPR      = 0.001     # flag at most 0.1 % of benign packets
SEED            = 0

assert DATASET_VERSION, "set DATASET_VERSION to the prepared dataset's version (it goes into the model metadata)"
manifest = json.loads( ( INPUT / "manifest.json" ).read_text( ) )
print( f"nla-ml { nla_ml.__version__ } @ { NLA_ML_COMMIT[ :7 ] }, scikit-learn { SKLEARN_VERSION }, features v{ FEATURE_VERSION } ({ FEATURE_FINGERPRINT })" )
print( f"input: { INPUT } (prepared with nla-ml @ { manifest[ 'nla_ml_commit' ][ :7 ] }, features v{ manifest[ 'feature_version' ] })" )

## Load train and val, deduplicated with counts

Each file is featurized here with the pinned nla-ml, so the features always match the version the model is exported with. The prepared files hold API entries, not features.

In [ ]:
# Capture day of each file. 8 and 15 are benign files between days; they go with the next day
DAY = { 1: "Mon", 2: "Mon", 3: "Tue", 4: "Tue", 5: "Tue", 6: "Tue", 7: "Tue", 8: "Wed", 9: "Wed", 10: "Wed",
        11: "Wed", 12: "Thu", 13: "Thu", 14: "Thu", 15: "Fri", 16: "Fri", 17: "Fri", 18: "Fri" }

def files_in( split: str ) -> list[ int ]:
    return sorted( int( n ) for n, s in manifest[ "splits" ].items( ) if s == split )

def load_deduplicated( split: str ) -> pd.DataFrame:
    """Featurize every file of a split; return one row per distinct (file, features, attack) with its packet count."""
    assert split in ( "train", "val" ), "02_train_rf must never read the test split"
    parts = [ ]
    for n in files_in( split ):
        t0 = time.perf_counter( )
        df = read_prepared( INPUT / f"file_{ n:02d}.parquet", columns=[ *ENTRY_COLUMNS, "attack" ] )
        X  = featurize_frame( df[ list( ENTRY_COLUMNS ) ] )
        X[ "attack" ] = df[ "attack" ].to_numpy( )
        rows = len( X )
        agg = X.value_counts( sort=False ).reset_index( name="count" )
        agg.insert( 0, "file", np.int8( n ) )
        parts.append( agg )
        print( f"  { split } file { n:2d}: { rows:>10,} packets -> { len( agg ):>9,} distinct rows [{ time.perf_counter( ) - t0:.0f}s]" )
        del df, X
        gc.collect( )
    out = pd.concat( parts, ignore_index=True )
    out[ "day" ]       = out[ "file" ].map( DAY )
    out[ "is_attack" ] = ( out[ "attack" ] != BENIGN_LABEL ).astype( "int8" )
    return out

train = load_deduplicated( "train" )
val   = load_deduplicated( "val" )
for name, d in ( ( "train", train ), ( "val", val ) ):
    print( f"{ name }: { d[ 'count' ].sum( ):,} packets in { len( d ):,} distinct rows "
           f"({ len( d ) / d[ 'count' ].sum( ):.1%}); attack packets { d.loc[ d.is_attack == 1, 'count' ].sum( ):,}" )

# How many val packets have a feature row that also occurs in train (repeated in 04_compare for test)
train_keys = set( pd.util.hash_pandas_object( train[ list( FEATURE_COLUMNS ) ], index=False ) )
val_seen   = pd.util.hash_pandas_object( val[ list( FEATURE_COLUMNS ) ], index=False ).isin( train_keys ).to_numpy( )
print( f"val packets whose feature row occurs in train: { val.loc[ val_seen, 'count' ].sum( ) / val[ 'count' ].sum( ):.1%}" )
del train_keys

## Helpers: weights, threshold, metrics (all packet-weighted)

In [ ]:
def packet_weights( d: pd.DataFrame ) -> np.ndarray:
    """count x balance: attack and benign packets carry equal total weight."""
    counts  = d[ "count" ].to_numpy( dtype=np.float64 )
    y       = d[ "is_attack" ].to_numpy( )
    per_cls = np.array( [ counts[ y == 0 ].sum( ), counts[ y == 1 ].sum( ) ] )
    return counts * ( counts.sum( ) / ( 2 * per_cls[ y ] ) )

def threshold_at_fpr( scores: np.ndarray, d: pd.DataFrame, fpr: float ) -> float:
    """Lowest threshold t such that benign packets with score > t are at most `fpr` of all benign packets."""
    benign = d[ "is_attack" ].to_numpy( ) == 0
    by_score = pd.Series( d[ "count" ].to_numpy( )[ benign ] ).groupby( scores[ benign ] ).sum( ).sort_index( ascending=False )
    over = by_score.index[ by_score.cumsum( ).to_numpy( ) > fpr * by_score.sum( ) ]
    return float( over[ 0 ] ) if len( over ) else -np.inf

def evaluate( scores: np.ndarray, d: pd.DataFrame, threshold: float ) -> dict:
    counts  = d[ "count" ].to_numpy( )
    y       = d[ "is_attack" ].to_numpy( )
    flagged = scores > threshold
    result  = {
        "benign_flagged_share": float( counts[ flagged & ( y == 0 ) ].sum( ) / max( counts[ y == 0 ].sum( ), 1 ) ),
        "alerts_per_million_benign": float( 1e6 * counts[ flagged & ( y == 0 ) ].sum( ) / max( counts[ y == 0 ].sum( ), 1 ) ),
        "attack_recall": float( counts[ flagged & ( y == 1 ) ].sum( ) / max( counts[ y == 1 ].sum( ), 1 ) ),
        "pr_auc": float( average_precision_score( y, scores, sample_weight=counts ) ) if 0 < y.sum( ) < len( y ) else None,
        "per_family": { },
    }
    for family, rows in d[ d[ "is_attack" ] == 1 ].groupby( "attack" ).groups.items( ):
        idx = d.index.get_indexer( rows )
        result[ "per_family" ][ family ] = {
            "packets": int( counts[ idx ].sum( ) ),
            "recall":  float( counts[ idx ][ flagged[ idx ] ].sum( ) / counts[ idx ].sum( ) ),
        }
    return result

def make_rf( params: dict, n_estimators: int, max_samples: float | None ) -> RandomForestClassifier:
    return RandomForestClassifier( n_estimators=n_estimators, max_samples=max_samples, n_jobs=-1, random_state=SEED, **params )

def fit( model: RandomForestClassifier, d: pd.DataFrame ) -> RandomForestClassifier:
    y = d[ "is_attack" ].to_numpy( ).astype( int )
    assert set( np.unique( y ) ) == { 0, 1 }, "training data needs both benign and attack rows"
    return model.fit( d[ list( FEATURE_COLUMNS ) ], y, sample_weight=packet_weights( d ) )

def attack_score( model: RandomForestClassifier, d: pd.DataFrame ) -> np.ndarray:
    return model.predict_proba( d[ list( FEATURE_COLUMNS ) ] )[ :, list( model.classes_ ).index( 1 ) ]

## Tune: leave-one-day-out CV

Each fold trains on four days and scores the fifth. For each config, out-of-fold scores are pooled across folds and thresholded at `TARGET_FPR`. In a fold, each family is mostly scored on a day whose attacks weren't in training, so these numbers lean towards *unseen-attack* performance and come out lower than `04_compare`'s seen-family results.

To keep it fast, CV uses fewer trees and bootstrap subsamples (`CV_TREES`, `CV_MAX_SAMPLES`). The final fit uses more.

In [ ]:
GRID = [
    { "max_depth": 8,  "max_features": "sqrt" },
    { "max_depth": 16, "max_features": "sqrt" },
    { "max_depth": 24, "max_features": "sqrt" },
    { "max_depth": 16, "max_features": 0.5 },
]
CV_TREES, CV_MAX_SAMPLES = 50, 0.25

days = sorted( train[ "day" ].unique( ), key=list( DAY.values( ) ).index )
print( "CV folds (held-out day: train files):", { day: sorted( train.loc[ train.day == day, "file" ].unique( ).tolist( ) ) for day in days } )

cv_rows = [ ]
for i, params in enumerate( GRID ):
    t0  = time.perf_counter( )
    oof = np.zeros( len( train ) )
    for day in days:
        held = ( train[ "day" ] == day ).to_numpy( )
        model = fit( make_rf( params, CV_TREES, CV_MAX_SAMPLES ), train[ ~held ] )
        oof[ held ] = attack_score( model, train[ held ] )
        del model
        gc.collect( )
    t = threshold_at_fpr( oof, train, TARGET_FPR )
    r = evaluate( oof, train, t )
    macro = float( np.mean( [ f[ "recall" ] for f in r[ "per_family" ].values( ) ] ) )
    cv_rows.append( { "config": i, **params, "macro_family_recall": macro, "attack_recall": r[ "attack_recall" ],
                      "pr_auc": r[ "pr_auc" ], **{ f"recall: { k }": v[ "recall" ] for k, v in r[ "per_family" ].items( ) },
                      "seconds": round( time.perf_counter( ) - t0 ) } )
    print( f"config { i } { params }: macro family recall { macro:.3f}, attack recall { r[ 'attack_recall' ]:.3f}, "
           f"PR-AUC { r[ 'pr_auc' ]:.3f} [{ cv_rows[ -1 ][ 'seconds' ] }s]" )

cv_results = pd.DataFrame( cv_rows )
cv_results.to_csv( OUT / "cv_results.csv", index=False )
# Best macro recall; ties go to the shallower (smaller, faster) model
best = cv_results.sort_values( [ "macro_family_recall", "max_depth" ], ascending=[ False, True ] ).iloc[ 0 ]
BEST_PARAMS = GRID[ int( best[ "config" ] ) ]
print( "chosen:", BEST_PARAMS )
display( cv_results.set_index( "config" ).T )

## Final fit and threshold on val

In [ ]:
FINAL_TREES, FINAL_MAX_SAMPLES = 100, 0.5

t0 = time.perf_counter( )
rf = fit( make_rf( BEST_PARAMS, FINAL_TREES, FINAL_MAX_SAMPLES ), train )
print( f"fitted { FINAL_TREES } trees in { time.perf_counter( ) - t0:.0f}s" )

val_scores = attack_score( rf, val )
THRESHOLD  = threshold_at_fpr( val_scores, val, TARGET_FPR )
val_report = evaluate( val_scores, val, THRESHOLD )
print( f"threshold { THRESHOLD:.4f} -> val benign flagged { val_report[ 'benign_flagged_share' ]:.4%} "
       f"({ val_report[ 'alerts_per_million_benign' ]:.0f} alerts per million benign packets)" )
print( "val per-family recall:", { k: round( v[ "recall" ], 3 ) for k, v in val_report[ "per_family" ].items( ) } )

importances = pd.Series( rf.feature_importances_, index=list( FEATURE_COLUMNS ) ).sort_values( ascending=False )
display( importances.round( 4 ).to_frame( "importance" ) )

## Export, reload like the backend, and time it

In [ ]:
model_version = f"cicids2017-v1-rf-{datetime.now( UTC ):%Y%m%d}"
metadata = export_model(
    rf, "rf", OUT / "models" / "rf",
    model_version=model_version,
    extra={
        "dataset":           manifest[ "dataset" ],
        "dataset_version":   DATASET_VERSION,
        "nla_ml_commit":     NLA_ML_COMMIT,
        "train_files":       files_in( "train" ),
        "val_files":         files_in( "val" ),
        "held_out_families": manifest[ "held_out_families" ],
        "params":            { **BEST_PARAMS, "n_estimators": FINAL_TREES, "max_samples": FINAL_MAX_SAMPLES },
        "weighting":         "deduplicated rows; sample_weight = packet count x class balance by packet count",
        "target_fpr":        TARGET_FPR,
        "threshold":         THRESHOLD,
        "val":               val_report,
        "cv":                { "folds": "leave-one-day-out", "trees": CV_TREES, "max_samples": CV_MAX_SAMPLES, "best_macro_family_recall": float( best[ "macro_family_recall" ] ) },
    },
)

# Load exactly as the backend does (checksum, versions, estimator, smoke test) and check it scores the same
loaded = load_model( OUT / "models" / "rf", "rf" )
check  = val.sample( min( 20_000, len( val ) ), random_state=SEED )
assert np.allclose( loaded.score( check[ list( FEATURE_COLUMNS ) ] ), attack_score( rf, check ) )

# Latency for one full API batch (500 entries): featurize + predict, single-threaded like the backend
raw = read_prepared( INPUT / f"file_{ files_in( 'val' )[ 0 ]:02d}.parquet", columns=list( ENTRY_COLUMNS ) )
batch = raw.sample( min( 500, len( raw ) ), random_state=SEED ).to_dict( orient="records" )
del raw
timings = [ ]
for _ in range( 30 ):
    t0 = time.perf_counter( )
    loaded.score( featurize( batch ) )
    timings.append( ( time.perf_counter( ) - t0 ) * 1000 )
latency_ms = float( np.median( timings[ 5: ] ) )
size_mb = ( OUT / "models" / "rf" / "model.joblib" ).stat( ).st_size / 1e6
print( f"{ model_version }: { size_mb:.1f} MB, { latency_ms:.1f} ms per 500-row batch (featurize + predict, 1 thread)" )
if latency_ms > 100:
    print( "WARNING: over ~100 ms per batch: revisit the thread-pool decision or use a smaller model" )

In [ ]:
report = {
    "model_version": model_version, "size_mb": round( size_mb, 2 ), "latency_ms_per_500": round( latency_ms, 1 ),
    "threshold": THRESHOLD, "target_fpr": TARGET_FPR, "params": metadata[ "params" ],
    "val": val_report, "feature_importances": importances.round( 5 ).to_dict( ),
    "cv": cv_results.to_dict( orient="records" ),
    "dataset_version": DATASET_VERSION, "nla_ml_commit": NLA_ML_COMMIT,
}
( OUT / "report.json" ).write_text( json.dumps( report, indent=2, default=str ) )
print( json.dumps( { k: report[ k ] for k in ( "model_version", "size_mb", "latency_ms_per_500", "threshold", "params" ) }, indent=2, default=str ) )

## Next

- Download: `kaggle kernels output hannana06/nla-02-train-rf -p ./artifacts`, then copy `artifacts/models/rf/` to the repo's `models/rf/`. It is git-ignored; only put artifacts you made there.
- `04_compare` evaluates this model on the **test** split: per family, packet- and flow-level, seen vs unseen, against the separability ceilings and the heuristics.